# 🎯 Repeat Purchase Prediction — Olist E-Commerce

**Author:** [Nama Kamu]
**Dataset:** Olist Brazilian E-Commerce
**Task:** Binary Classification — Prediksi apakah customer akan repeat purchase

---

## 🎯 Business Problem

Repeat purchase rate Olist hanya ~3% [citation:6]. 
Artinya, 97% customer hanya belanja **sekali** lalu hilang.

**Pertanyaan:** 
"Customer mana yang berpotensi repeat purchase, sehingga marketing bisa 
menargetkan mereka dengan insentif yang tepat?"

## 📌 Tantangan Utama

1. **Class Imbalance Ekstrem:** Hanya ~3% positive class (repeat buyer)
2. **Data Leakage Risk:** Fitur seperti `recency` atau `frequency` bisa 
   "membocorkan" label jika tidak di-handle dengan temporal split
3. **Metric Selection:** Accuracy akan menyesatkan — gunakan **PR-AUC** [citation:4]

## 🛠️ Tools
Python, Pandas, Scikit-Learn, XGBoost, Imbalanced-Learn (SMOTE)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve)
from xgboost import XGBClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
print("✅ Library loaded")

✅ Library loaded


In [2]:
import os

RAW = os.path.join('..', 'data', 'raw')

# Load tabel
orders = pd.read_csv(os.path.join(RAW, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(RAW, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(RAW, 'olist_customers_dataset.csv'))
products = pd.read_csv(os.path.join(RAW, 'olist_products_dataset.csv'))

# Filter delivered
orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Gabung
df = order_items.merge(
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
    on='order_id', how='inner'
).merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='inner'
).merge(
    products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)

df['revenue'] = df['price'] + df['freight_value']

print(f"✅ Data loaded: {df.shape[0]:,} baris")
print(f"👥 Unique customers: {df['customer_unique_id'].nunique():,}")
print(f"📅 Periode: {df['order_purchase_timestamp'].min().date()} → {df['order_purchase_timestamp'].max().date()}")

✅ Data loaded: 110,197 baris
👥 Unique customers: 93,358
📅 Periode: 2016-09-15 → 2018-08-29


In [3]:
# ===== DEFINISI TARGET =====
# Tujuan: Prediksi apakah customer akan repeat purchase
# dalam jendela waktu tertentu SETELAH first purchase.

# 1. Identifikasi first purchase per customer
first_purchase = df.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index()
first_purchase.columns = ['customer_unique_id', 'first_purchase_date']

# 2. Tentukan cut-off: analisis hanya customer yang first purchase-nya
#    minimal 180 hari sebelum akhir data (biar ada waktu observasi)
max_date = df['order_purchase_timestamp'].max()
cutoff_date = max_date - pd.Timedelta(days=180)

eligible_customers = first_purchase[first_purchase['first_purchase_date'] <= cutoff_date].copy()
print(f"👥 Customer eligible (first purchase ≤ {cutoff_date.date()}): {len(eligible_customers):,}")

# 3. Tentukan label: repeat purchase dalam 180 hari setelah first purchase
df_eligible = df.merge(eligible_customers[['customer_unique_id', 'first_purchase_date']],
                        on='customer_unique_id', how='inner')

df_eligible['days_since_first'] = (df_eligible['order_purchase_timestamp'] -
                                    df_eligible['first_purchase_date']).dt.days

# Repeat = ada order lain dalam 180 hari setelah first purchase
repeat_customers = df_eligible[
    (df_eligible['days_since_first'] > 0) &
    (df_eligible['days_since_first'] <= 180)
]['customer_unique_id'].unique()

eligible_customers['is_repeat'] = eligible_customers['customer_unique_id'].isin(repeat_customers).astype(int)

print(f"\n📊 Distribusi Target:")
print(eligible_customers['is_repeat'].value_counts())
print(f"\n⚠️  Repeat rate: {eligible_customers['is_repeat'].mean()*100:.2f}%")
print(f"   → Class imbalance EKSTREM! Gunakan PR-AUC sebagai metric utama.")

👥 Customer eligible (first purchase ≤ 2018-03-02): 55,907

📊 Distribusi Target:
is_repeat
0    54812
1     1095
Name: count, dtype: int64

⚠️  Repeat rate: 1.96%
   → Class imbalance EKSTREM! Gunakan PR-AUC sebagai metric utama.


In [ ]:
# ===== FEATURE ENGINEERING =====
# ⚠️ PENTING: Fitur harus dihitung HANYA dari data first purchase
# (bukan seluruh riwayat) untuk menghindari data leakage.

# Ambil data first purchase per customer
first_order_data = df_eligible[df_eligible['days_since_first'] == 0].copy()

# Agregasi fitur per customer dari first purchase
features = first_order_data.groupby('customer_unique_id').agg(
    # Fitur transaksi
    first_order_revenue=('revenue', 'sum'),
    first_order_items=('order_id', 'count'),
    first_order_products=('product_id', 'nunique'),
    first_order_categories=('product_category_name', 'nunique'),
    avg_price=('price', 'mean'),
    avg_freight=('freight_value', 'mean'),
    
    # Fitur geografis
    customer_state=('customer_state', 'first'),
    
    # Fitur waktu (dari first purchase saja — TIDAK BOLEH pakai recency!)
    first_purchase_month=('order_purchase_timestamp', lambda x: x.dt.month.iloc[0]),
    first_purchase_dow=('order_purchase_timestamp', lambda x: x.dt.dayofweek.iloc[0]),
).reset_index()

# Merge dengan label
model_data = features.merge(eligible_customers[['customer_unique_id', 'is_repeat']],
                             on='customer_unique_id', how='inner')

print(f"✅ Model dataset: {model_data.shape[0]:,} customers, {model_data.shape[1]} kolom")
print(f"📊 Target distribution:")
print(model_data['is_repeat'].value_counts(normalize=True).round(3))

In [ ]:
# ===== ENCODING & SPLIT =====
# Pisahkan fitur & target
X = model_data.drop(columns=['customer_unique_id', 'is_repeat'])
y = model_data['is_repeat']

# One-hot encoding untuk categorical
X = pd.get_dummies(X, columns=['customer_state'], drop_first=True)

# Train-test split (stratified untuk jaga proporsi kelas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"📊 Train set: {X_train.shape[0]:,} samples")
print(f"📊 Test set:  {X_test.shape[0]:,} samples")
print(f"\n📊 Proporsi target di train:")
print(y_train.value_counts(normalize=True).round(4))
print(f"\n📊 Proporsi target di test:")
print(y_test.value_counts(normalize=True).round(4))

In [ ]:
# ===== BASELINE: Logistic Regression =====
# Scaling untuk LR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate
print("=" * 60)
print("📊 BASELINE — LOGISTIC REGRESSION")
print("=" * 60)
print(f"\nROC-AUC:  {roc_auc_score(y_test, y_proba_lr):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_proba_lr):.4f}  ← METRIC UTAMA")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['No Repeat', 'Repeat']))

In [ ]:
# ===== HANDLING CLASS IMBALANCE =====
# Strategi 1: Class Weight (tanpa resampling)
# Strategi 2: SMOTE (synthetic oversampling)

# --- XGBoost dengan Class Weight ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_weighted = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,  # 👈 handle imbalance
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    use_label_encoder=False
)
xgb_weighted.fit(X_train, y_train)

y_proba_xgb_w = xgb_weighted.predict_proba(X_test)[:, 1]
y_pred_xgb_w = xgb_weighted.predict(X_test)

print("=" * 60)
print("📊 XGBOOST + CLASS WEIGHT")
print("=" * 60)
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_xgb_w):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_proba_xgb_w):.4f}")

In [ ]:
# ===== PERBANDINGAN MODEL =====
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost + Class Weight', 'XGBoost + SMOTE'],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_xgb_w),
        roc_auc_score(y_test, y_proba_xgb_s)
    ],
    'PR-AUC': [
        average_precision_score(y_test, y_proba_lr),
        average_precision_score(y_test, y_proba_xgb_w),
        average_precision_score(y_test, y_proba_xgb_s)
    ]
}).round(4)

print("=" * 60)
print("📊 PERBANDINGAN MODEL")
print("=" * 60)
print(results.to_string(index=False))

# ===== THRESHOLD TUNING =====
# Bisnis: kita mau precision tinggi (jangan salah target customer)
# tapi tetap tangkap cukup banyak repeat buyer
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_xgb_s)

# Cari threshold dengan F1 terbaik
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\n🎯 Best Threshold (F1-optimal): {best_threshold:.3f}")
print(f"   Precision: {precision[best_idx]:.3f}")
print(f"   Recall:    {recall[best_idx]:.3f}")
print(f"   F1:        {f1_scores[best_idx]:.3f}")

In [ ]:
# ===== VISUALISASI =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Precision-Recall Curve
for name, proba in [('LR', y_proba_lr), ('XGB+Weight', y_proba_xgb_w), ('XGB+SMOTE', y_proba_xgb_s)]:
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[0].plot(r, p, label=f'{name} (AP={ap:.3f})', linewidth=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Confusion Matrix (XGB+SMOTE dengan threshold optimal)
y_pred_optimal = (y_proba_xgb_s >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Repeat', 'Repeat'],
            yticklabels=['No Repeat', 'Repeat'])
axes[1].set_title(f'Confusion Matrix (threshold={best_threshold:.2f})', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('../reports/figures/model_pr_curve_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ===== FEATURE IMPORTANCE (XGBoost) =====
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_smote.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Feature Importance — XGBoost + SMOTE', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("=" * 60)
print("📊 TOP 10 FEATURE IMPORTANCE")
print("=" * 60)
for i, row in importance.head(10).iterrows():
    print(f"{row['feature']:30s} : {row['importance']:.4f}")

In [ ]:
print("=" * 70)
print("📌 RINGKASAN — REPEAT PURCHASE PREDICTION")
print("=" * 70)
print(f"""
📊 DATASET & TARGET
- Total customer eligible: {len(model_data):,}
- Repeat rate: {y.mean()*100:.2f}% (class imbalance ekstrem)
- Cut-off: 180 hari observasi

🤖 MODEL PERFORMANCE
- PR-AUC terbaik: {results['PR-AUC'].max():.4f} (XGBoost + SMOTE)
- Baseline (random): {y.mean():.4f}
- Lift: {results['PR-AUC'].max() / y.mean():.1f}x di atas random

📌 CATATAN PENTING
PR-AUC ~0.10 pada dataset ini adalah WAJAR [citation:4].
Repeat rate Olist memang sangat rendah (~3%) [citation:6].
Model tidak dimaksudkan untuk prediksi sempurna, tapi untuk
PRIORITISASI customer mana yang layak dapat insentif.

💡 REKOMENDASI BISNIS
1. Gunakan model untuk ranking customer (bukan hard yes/no)
2. Target top 10-20% customer dengan probability tertinggi
3. Berikan insentif second-purchase (diskon, free shipping)
4. A/B test: bandingkan conversion rate target vs non-target

🚀 NEXT STEPS
- Deploy model sebagai scoring API / batch scoring
- Dashboard Streamlit untuk visualisasi risk score
- Integrasi dengan CRM/marketing automation
""")
print("=" * 70)